In [28]:
from pathlib import Path
import sys
BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR))

In [29]:
file_path = [
    f"{BASE_DIR}/jars/delta-spark_2.12-3.1.0.jar",
    f"{BASE_DIR}/jars/delta-storage-3.1.0.jar",
    f"{BASE_DIR}/jars/gcs-connector-hadoop3-2.2.22-shaded.jar"
]
JARS = ",".join(file_path)

In [30]:
from src.utils import get_spark
from pyspark.sql import functions as F
spark = get_spark(jars = JARS, key = BASE_DIR/"key.json")

In [31]:
df_orders = spark.read.parquet("gs://instacart-platform/bronze/orders/*.parquet")
df_order_products__train = spark.read.parquet("gs://instacart-platform/bronze/order_products__train/*.parquet")
df_order_products__prior = spark.read.parquet("gs://instacart-platform/bronze/order_products__prior/*.parquet")
df_aisles = spark.read.parquet("gs://instacart-platform/bronze/aisles/*.parquet")
df_departments = spark.read.parquet("gs://instacart-platform/bronze/departments/*.parquet")
df_products = spark.read.parquet("gs://instacart-platform/bronze/products/*.parquet")

In [32]:
df_order_products__train.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [33]:
df_order_products__prior.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [52]:
df_order_products = df_order_products__prior.union(df_order_products__train)

In [53]:
df_order_products.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [54]:
df_order_products.groupBy('product_id').agg(F.count("*").alias("total")).filter("total > 1").show()

+----------+-----+
|product_id|total|
+----------+-----+
|     41890| 2014|
|      8592| 1023|
|     29228|  567|
|     40386| 2831|
|     42834| 1903|
|      8638| 4290|
|     18498|  743|
|     28836| 5217|
|       148| 5000|
|     40574|  519|
|      9852|  604|
|     17389|  481|
|     38153|  129|
|     38395|  968|
|     31236|  678|
|     37489| 1447|
|     22346|   41|
|      5518|   47|
|     43688|   36|
|     34234| 6799|
+----------+-----+
only showing top 20 rows



In [55]:
df_order_products.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)for c in df_order_products.columns]).show(5)

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       0|         0|                0|        0|
+--------+----------+-----------------+---------+



In [36]:
df_orders.show(5)

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
+--------+-------+--------+------------+---------+-----------------+----------------------+
only showing top 5 rows



In [ ]:
df_orders.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)for c in df_orders.columns]).show(5)

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+----------------------+



In [38]:
df_orders.count()

3421083

In [39]:
df_orders.filter(F.col("days_since_prior_order").isNull()).select("user_id").count()

206209

In [40]:
df_orders.groupBy("user_id").agg(F.count(F.when(F.col("days_since_prior_order").isNull(), 1)).alias("null_count")).filter(F.col("null_count") > 1).show(5)

+-------+----------+
|user_id|null_count|
+-------+----------+
+-------+----------+



In [41]:
df_orders.groupBy("order_id").count().filter("count > 1").show(5)

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [42]:
df_orders.groupBy("user_id").count().orderBy(F.desc("count")).filter("count > 1").show(5)

+-------+-----+
|user_id|count|
+-------+-----+
|  16680|  100|
|  31674|  100|
|  15221|  100|
|    310|  100|
|   2231|  100|
+-------+-----+
only showing top 5 rows



In [43]:
df_products = (df_products.join(df_aisles, on="aisle_id", how="left").join(df_departments, on="department_id", how="left"))

In [44]:
df_products.show(5)

+-------------+--------+----------+--------------------+--------------------+----------+
|department_id|aisle_id|product_id|        product_name|               aisle|department|
+-------------+--------+----------+--------------------+--------------------+----------+
|           19|      61|         1|Chocolate Sandwic...|       cookies cakes|    snacks|
|           13|     104|         2|    All-Seasons Salt|   spices seasonings|    pantry|
|            7|      94|         3|Robust Golden Uns...|                 tea| beverages|
|            1|      38|         4|Smart Ones Classi...|        frozen meals|    frozen|
|           13|       5|         5|Green Chile Anyti...|marinades meat pr...|    pantry|
+-------------+--------+----------+--------------------+--------------------+----------+
only showing top 5 rows



In [45]:
df_products.filter(F.col('department_id').isNull()).count()

0

In [46]:
df_products.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)for c in df_products.columns]).show(5)

+-------------+--------+----------+------------+-----+----------+
|department_id|aisle_id|product_id|product_name|aisle|department|
+-------------+--------+----------+------------+-----+----------+
|            0|       0|         0|           0|    1|         1|
+-------------+--------+----------+------------+-----+----------+



In [47]:
df_products.filter(F.col('aisle').isNull() | F.col('department').isNull()).show(5)

+-------------+--------+----------+--------------------+-----+----------+
|department_id|aisle_id|product_id|        product_name|aisle|department|
+-------------+--------+----------+--------------------+-----+----------+
|         Red"| Blunted|      6816|"Scotch Kids 5"" ...| NULL|      NULL|
+-------------+--------+----------+--------------------+-----+----------+



In [48]:
df_products.groupBy(df_products.columns).count().filter("count > 1").show(5)

+-------------+--------+----------+------------+-----+----------+-----+
|department_id|aisle_id|product_id|product_name|aisle|department|count|
+-------------+--------+----------+------------+-----+----------+-----+
+-------------+--------+----------+------------+-----+----------+-----+



In [49]:
df_products.groupBy("product_name").count().filter("count > 1").show(5)

+------------+-----+
|product_name|count|
+------------+-----+
+------------+-----+

